In [3]:
import pandas as pd
import sys
sys.path.append("..")
import main
import os

In [4]:
dataset_names = ["brca_n", "Facebook", "polluted_synthetic", "synthetic"]
datasets = {name: main.get_data(name) for name in dataset_names}
targets = {name : main.get_target(name) for name in dataset_names}


In [5]:
target_entries = {d: datasets[d][targets[d][0]] == targets[d][1] for d in datasets}

In [10]:
df = pd.DataFrame(columns=["Dataset", "Descriptions", "Time taken (s)"])
for d in datasets:
    results = pd.read_csv(os.path.join("results", f"{d}.csv"))
    descriptions = results["subgroup"].to_list()
    descriptions = [desc.split(" AND ") for desc in descriptions]
    time_file = open(os.path.join("results", f"{d}_time.txt"), "r")
    time_taken = time_file.readline()
    time_taken = float(time_taken.strip().split(": ")[1])
    df.loc[len(df)] = [d, descriptions, time_taken]

interpretability = pd.DataFrame(columns = ["Model", "Dataset", "# Subgroups", "Avg. size", "Maximum Jaccard"])


In [11]:
df_with_youden = df.copy()
for index, row in df.iterrows():
    global_entry = pd.Series(False, index = datasets[row["Dataset"]].index)
    local_entries = []
    maximum_jaccard = 0
    subgroups = 0
    sizes = []
    for subgroup in row["Descriptions"]:
        entry = pd.Series(True, index = datasets[row["Dataset"]].index)
        for selector in subgroup:
            attr, val = selector.split("==")
            val = val.strip().strip("'").strip('"')
            entry = entry & (datasets[row["Dataset"]][attr] == val)
        for local_entry in local_entries:
            jaccard = sum(entry & local_entry) / sum(entry | local_entry)
            maximum_jaccard = max(maximum_jaccard, jaccard)
        local_entries.append(entry)
        subgroups += 1
        sizes.append(len(subgroup))
        global_entry = global_entry | entry
    positive = sum(global_entry & target_entries[row["Dataset"]]) / sum(target_entries[row["Dataset"]])
    negative = sum(global_entry & ~target_entries[row["Dataset"]]) / sum(~target_entries[row["Dataset"]])
    youden = positive - negative
    df_with_youden.at[index, "Youden"] = youden
    interpretability.loc[len(interpretability)] = ["Beam", row["Dataset"], subgroups, sum(sizes) / subgroups, maximum_jaccard]
df = df_with_youden

In [12]:
df

,Dataset,Descriptions,Time taken (s),Youden
0,brca_n,"[[ENSG00000258227=='1.0'], [ENSG00000205629=='...",24.499520,0.963964
1,Facebook,"[[2032=='0', 4641=='0', 4686=='0'], [2032=='0'...",953.918057,0.292973
2,polluted_synthetic,"[[Attribute 4=='1', Attribute 5=='1'], [Attrib...",382.169393,0.785816
3,synthetic,"[[Attribute 4=='1', Attribute 5=='1'], [Attrib...",993.282024,1.000000


In [13]:
interpretability

,Model,Dataset,# Subgroups,Avg. size,Maximum Jaccard
0,Beam,brca_n,10,1.00,0.577465
1,Beam,Facebook,3,3.00,0.928769
2,Beam,polluted_synthetic,20,2.65,1.000000
3,Beam,synthetic,50,2.78,1.000000
